# Plot windowed TCC, RMSE, or bias maps

Plots selected cached map products across experiments and lead windows.


In [ ]:
from pathlib import Path
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_out")

group = "Jan2012"
run = "fc"
obs = "ERA5"
variable = "TREFHT"
metric = "TCC"
windows = None  # None plots every window in the selected scheme
mask = "none"
vmin, vmax = 0.0, 1.0
levels = None
show = True


window_scheme = "biweekly"  # "biweekly", "weekly", "15day", or "month"


In [ ]:
from pathlib import Path
import sys

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))

from configs.output_paths import workflow_output_dirs
from configs.s2s_config import EXPERIMENT_GROUPS, OBS_VARIABLES, PCC_REGIONS, TCC_REGIONS, LEAD_WINDOWS, WEEKLY_WINDOWS, FIFTEEN_DAY_WINDOWS, MONTH_WINDOWS, WINDOW_SCHEMES, AVAILABLE_PCC_METRICS, AVAILABLE_PCC_MAP_METRICS, AVAILABLE_TCC_METRICS
from util.s2s_experiments import build_experiments
from util.s2s_observations import OBS_REGISTRY, get_obs_file


## Notes: available selections

- Experiment groups: Dec2011 (analysis), Jan2012, and Jun2012 (forecasts).
- Runs: da for Dec2011; fc (standard forecast) or wc (water-cycle forecast) for forecast groups.
- Observations and variables: `ERA5` supports UBOT, VBOT, PRECT, PSL, T925, Q925, U850, V850, T850, Q850, Z500, OMEGA850/500/200, U/V500, U/V200, FLDS/C, FLNS/C, FSDS/C, FSNS/C, LHFLX, SHFLX, QFLX, TREFHT, TS, PRECSL, PRECC, and PRECL; `GPCP` and `GPM` support PRECT; `NOAA-OLR` supports FLUT.
- Regions: GLOBAL, land/ocean where applicable, NINO3.4, MAR_CONT, CONUS, NH_POLAR.
- Existing NetCDF diagnostics are reused. Set `force_compute = True` to rebuild selected products.

Available metrics: `TCC`, `RMSE`, `BIAS`. Mask choices: `none`, `land`, `ocean`.

Exact base-map fields are `TCC`, `RMSE`, and `BIAS` for members and `TCC_ensmean`, `RMSE_ensmean`, and `BIAS_ensmean` for the ensemble mean. For each base metric, regional outputs provide `<METRIC>_regional_member`, `<METRIC>_member_ens_mean`, `<METRIC>_member_ens_std`, `<METRIC>_member_qpostXX`, `<METRIC>_member_qpreXX`, `<METRIC>_regional_ensmean`, valid-weight and valid-fraction fields, plus `region_area_weight_sum`. `XX` is each configured quantile percentile.


In [ ]:
from util.s2s_tcc_map_plotting import TccWindowMapPlotConfig, TccWindowMapPlotter

base_dir, figure_dir = workflow_output_dirs("fcst_s2s_tcc", f"{window_scheme}_metrics", output_root=DIAGNOSTIC_OUTPUT_ROOT)
meta=EXPERIMENT_GROUPS[group]; exp_dict=build_experiments(str(INPUT_DATA_ROOT)); models=meta["models"]
plotter=TccWindowMapPlotter(TccWindowMapPlotConfig(input_dir=base_dir/group, group=group, freq=meta["freq"], run=run, obs=obs, period=meta["period"], exp_order=models, nens={e:int(exp_dict[e]["nens"]) for e in models}, lead_bins=WINDOW_SCHEMES[window_scheme], cmap="RdBu_r" if metric.upper()=="BIAS" else "viridis", vmin=vmin, vmax=vmax, mask=mask, mask_file=INPUT_DATA_ROOT/"reference/lnd_sea_mask/landmask_1x1.nc"))
windows = tuple(WINDOW_SCHEMES[window_scheme]) if windows is None else tuple(windows)
out=figure_dir/f"{group}_{run}_{obs}_{variable}_{metric}_{window_scheme}_windows.png"
plotter.plot_var_panel_exps_x_windows(var=variable, unit="", exps=models, windows=windows, metric=metric, var_levels=levels, out_png=out, show=show)

